## 0. Import

In [ ]:
# # DOWNLOAD DATASET 
# import kagglehub

# # Thay đường dẫn thư mục ở đây
# target_dir = r"D:\project\deepnn\forecast-practice\datasets\mnist-multi-digit-regression-dataset"

# path = kagglehub.dataset_download(
#     "youssef019/mnist-multi-digit-regression-dataset",
#     output_dir=target_dir
# )

# print("Path to dataset files:", path)

In [2]:
import numpy as np
import pandas as pd
from PIL import Image
from math import sqrt
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, Permute, Reshape, Dense,
                                     Embedding, LSTM, Attention, Concatenate,
                                     GlobalAveragePooling1D, BatchNormalization, Dropout)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [ ]:
IMG_H, IMG_W = 64, 128
PAD_TOKEN = 10
START_TOKEN = 11                  # token bắt đầu cho decoder
N_CLASSES = 11              # 0-9 và PAD
EPOCHS = 50
BATCH_SIZE = 64

## 1. Read dataset

In [9]:
df = pd.read_csv(r'D:\project\deepnn\forecast-practice\datasets\labels.csv', dtype=str) # đọc dạng chuỗi để không mất số 0 ở đầu nhãn
print(df.head(), end="\n\n")
print(df.info())

   image     label
0  0.png   6290448
1  1.png    101651
2  2.png  36731246
3  3.png  24680373
4  4.png   8868017

<class 'pandas.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   image   60000 non-null  str  
 1   label   60000 non-null  str  
dtypes: str(2)
memory usage: 937.6 KB
None


In [ ]:
images = []
labels = []

for image in df['image']:
    # thay đường dân folder ảnh vào đây
    image_path = r"D:\project\deepnn\forecast-practice\datasets\images\\" + image 

    with Image.open(image_path) as img:
        img = img.convert('L')
        img = img.resize((IMG_W, IMG_H))
        img_arr = np.array(img, dtype=np.float32) / 255 # chuyển ảnh thành ma trận số và scale về [0, 1]
        images.append(img_arr)

for label in df['label']:
    labels.append([int(digit) for digit in label])
    
max_len = max(len(label) for label in labels)

print("Number of images:", len(images))
print("Image shape:", images.shape)
print("Maximum label length:", max_len)

KeyboardInterrupt: 

tối đa 8 chữ số label

## 2. Preprocessing

In [ ]:
# Thêm chiều kênh màu (N, 64, 128, 1) cho mạng CNN
images = np.expand_dims(np.array(images), axis=-1)

In [ ]:
# Padding nhãn cho Encoder
y = np.full((len(labels), max_len), PAD_TOKEN, dtype='int32')
for i, seq in enumerate(labels):
    y[i, :len(seq)] = seq

In [ ]:
# Tạo đầu vào cho Decoder:
# Dịch nhãn sang phải 1 bước và chèn START_TOKEN vào đầu
start_col = np.full((len(y), 1), START_TOKEN, dtype='int32')
dec_in = np.concatenate([start_col, y[:, :-1]], axis=1)

## 3. Data Split

Chia train/val/test = 70/15/15

In [ ]:
idx = np.arange(len(images))
idx_train, idx_tmp = train_test_split(idx, test_size=0.30, random_state=42)
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.50, random_state=42)

X_train, X_val, X_test = images[idx_train], images[idx_val], images[idx_test]
D_train, D_val, D_test = dec_in[idx_train], dec_in[idx_val], dec_in[idx_test]
y_train, y_val, y_test = y[idx_train], y[idx_val], y[idx_test]

print('Train:', X_train.shape)
print('Validation:', X_val.shape)
print('Test:', X_test.shape)